In [1]:
# Dependências usadas neste notebook. Rode esta célula uma vez antes do restante.
# Se já instalou via requirements.txt do projeto, pode pular.
%pip install --quiet pandas pyarrow geopandas

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Diagnóstico: por que existem setores sem renda?

Este notebook investiga, **sem alterar a pipeline**, as causas de `tem_renda=0` (e de `renda_v06004` NULL com `tem_renda=1`) na base final gerada por `pipeline_do_zero_cnefe_setor_cep_renda.ipynb`.

Lembrete: a `final_query` da pipeline tem como driver a tabela `pares_logradouro` (CNEFE) e faz `LEFT JOIN` com `renda` e `setores`. Logo, o output só contém setores que aparecem no CNEFE.

Trabalhamos com **três universos**:

- **A** — setores oficiais do shapefile `BR_setores_CD2022`
- **B** — setores com linha no CSV de renda (subdivididos em B* = com `V06004` numérico)
- **C** — setores presentes no CNEFE (tabela `pares_logradouro` do SQLite de trabalho)

Toda análise abaixo é **somente leitura** dos arquivos já existentes.

Por padrão filtramos por **SP** (cod_uf=`35`), que é o recorte que está hoje no SQLite/Parquet de trabalho.

## Roteiro

1. **Etapa 1** — Quantificar o problema no output atual (Parquet/SQLite).
2. **Etapa 2** — Diagrama de Venn dos conjuntos A, B, B*, C.
3. **Etapa 3** — Caracterizar `C − B` (setores no CNEFE fora do CSV de renda): municípios, endereços impactados.
4. **Etapa 4** — Carregar `CD_TIPO` do shapefile e cruzar com C−B e B−B* para entender concentração por tipo de setor.
5. **Etapa 5** — Inspecionar o CSV de renda cru para ver como o IBGE marca sigilo estatístico.
6. **Etapa 6** — Síntese: tabela de causas e contagens.

## Setup — imports, caminhos e filtro de UF

In [2]:
from pathlib import Path
import sqlite3

import pandas as pd
from IPython.display import display

try:
    import geopandas as gpd
    GEOPANDAS_AVAILABLE = True
except Exception:
    gpd = None
    GEOPANDAS_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "saida_cnefe_uf").exists() and (candidate / "BR_setores_CD2022").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
RENDA_CSV = PROJECT_ROOT / "Agregados_por_setores_renda_responsavel_BR_csv" / "Agregados_por_setores_renda_responsavel_BR.csv"
SHAPEFILE = PROJECT_ROOT / "BR_setores_CD2022" / "BR_setores_CD2022.shp"
WORK_SQLITE = PROJECT_ROOT / "saida_setor_cep_renda_do_zero" / "setor_cep_renda_do_zero_work.sqlite"
OUT_PARQUET = PROJECT_ROOT / "saida_setor_cep_renda_do_zero" / "setor_censitario_cep_renda_do_zero.parquet"

# Recorte do diagnóstico. Deve casar com o que está no SQLite/Parquet atuais.
UF_COD = "35"  # SP
UF_SIGLA = "SP"

for path in [RENDA_CSV, SHAPEFILE, WORK_SQLITE, OUT_PARQUET]:
    print(f"{path.name}: {'OK' if path.exists() else 'NAO ENCONTRADO'}  ({path})")

Agregados_por_setores_renda_responsavel_BR.csv: OK  (C:\Users\matpa\Documents\Base CEP_LOG_RENDA\Agregados_por_setores_renda_responsavel_BR_csv\Agregados_por_setores_renda_responsavel_BR.csv)
BR_setores_CD2022.shp: OK  (C:\Users\matpa\Documents\Base CEP_LOG_RENDA\BR_setores_CD2022\BR_setores_CD2022.shp)
setor_cep_renda_do_zero_work.sqlite: OK  (C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_setor_cep_renda_do_zero\setor_cep_renda_do_zero_work.sqlite)
setor_censitario_cep_renda_do_zero.parquet: OK  (C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_setor_cep_renda_do_zero\setor_censitario_cep_renda_do_zero.parquet)


## Etapa 1 — Quantificar o problema no output atual

Carregamos o Parquet final e medimos:

- pares e setores distintos com `tem_renda=0`
- pares e setores com `tem_renda=1` mas `renda_v06004` NULL (sigilo disfarçado)
- impacto em `qtd_enderecos_setor_cep`

In [3]:
final = pd.read_parquet(OUT_PARQUET)
print(f"Linhas no parquet final: {len(final):,}")
print(f"Setores distintos: {final['cd_setor'].nunique():,}")
print(f"CEPs distintos:    {final['cep'].nunique():,}")
print(f"Total de endereços agregados: {final['qtd_enderecos_setor_cep'].sum():,}")
display(final.head(3))

Linhas no parquet final: 589,195
Setores distintos: 100,723
CEPs distintos:    275,658
Total de endereços agregados: 22,953,725


,cd_setor,cep,cod_uf,sigla_uf,nm_uf,cod_municipio,nm_municipio,area_km2,cep_inicial_setor,cep_final_setor,faixa_cep_setor,qtd_ceps_no_setor,qtd_setores_no_cep,qtd_enderecos_setor_cep,qtd_logradouros_distintos_setor_cep,total_enderecos_no_setor,total_enderecos_no_cep,pct_enderecos_do_setor_no_cep,pct_enderecos_do_cep_no_setor,tem_renda,renda_v06001,renda_v06002,renda_v06003,renda_v06004,renda_v06005
0,350010505000001,17800015,35,SP,São Paulo,3500105,Adamantina,0.1238,17800009,17800049,17800009 - 17800049,9,2,118,1,530,164,0.2226,0.7195,1,138.0000,288.0000,1.4400,"4,107.2900","24,164,514.2500"
1,350010505000001,17800011,35,SP,São Paulo,3500105,Adamantina,0.1238,17800009,17800049,17800009 - 17800049,9,3,98,1,530,175,0.1849,0.5600,1,138.0000,288.0000,1.4400,"4,107.2900","24,164,514.2500"
2,350010505000001,17800047,35,SP,São Paulo,3500105,Adamantina,0.1238,17800009,17800049,17800009 - 17800049,9,3,77,1,530,156,0.1453,0.4936,1,138.0000,288.0000,1.4400,"4,107.2900","24,164,514.2500"


In [4]:
has_renda_row = final["tem_renda"].eq(1)
v06004_null = final["renda_v06004"].isna()

metrics = pd.DataFrame({
    "categoria": [
        "tem_renda = 0 (setor ausente do CSV de renda)",
        "tem_renda = 1 e renda_v06004 numerico",
        "tem_renda = 1 e renda_v06004 NULL (sigilo disfarçado)",
    ],
    "pares": [
        int((~has_renda_row).sum()),
        int((has_renda_row & ~v06004_null).sum()),
        int((has_renda_row & v06004_null).sum()),
    ],
    "setores_distintos": [
        final.loc[~has_renda_row, "cd_setor"].nunique(),
        final.loc[has_renda_row & ~v06004_null, "cd_setor"].nunique(),
        final.loc[has_renda_row & v06004_null, "cd_setor"].nunique(),
    ],
    "enderecos": [
        int(final.loc[~has_renda_row, "qtd_enderecos_setor_cep"].sum()),
        int(final.loc[has_renda_row & ~v06004_null, "qtd_enderecos_setor_cep"].sum()),
        int(final.loc[has_renda_row & v06004_null, "qtd_enderecos_setor_cep"].sum()),
    ],
})
total_pares = metrics["pares"].sum()
total_end = metrics["enderecos"].sum()
metrics["pct_pares"] = (metrics["pares"] / total_pares * 100).round(2)
metrics["pct_enderecos"] = (metrics["enderecos"] / total_end * 100).round(2)
display(metrics)

,categoria,pares,setores_distintos,enderecos,pct_pares,pct_enderecos
0,tem_renda = 0 (setor ausente do CSV de renda),41470,9626,1624792,7.0400,7.0800
1,tem_renda = 1 e renda_v06004 numerico,545648,89971,21307605,92.6100,92.8300
2,tem_renda = 1 e renda_v06004 NULL (sigilo disf...,2077,1126,21328,0.3500,0.0900


## Etapa 2 — Diagrama de Venn dos conjuntos A, B, C

Construímos os três conjuntos para SP:

- **A**: `cd_setor` do shapefile com `CD_UF = '35'`
- **B**: `cd_setor` do CSV de renda começando com `35`
- **B***: subconjunto de B com `V06004` numérico (renda média preenchida)
- **C**: `cd_setor` distintos em `pares_logradouro` (SQLite)

Em seguida computamos tamanhos e diferenças.

In [5]:
if not GEOPANDAS_AVAILABLE:
    raise RuntimeError("geopandas necessário para ler atributos do shapefile.")

shp_attrs_full = gpd.read_file(SHAPEFILE, ignore_geometry=True)
print("Colunas do shapefile:")
print(list(shp_attrs_full.columns))
print(f"\nTotal de setores no shapefile (Brasil): {len(shp_attrs_full):,}")
shp_sp = shp_attrs_full.loc[shp_attrs_full["CD_UF"].astype(str).str.zfill(2) == UF_COD].copy()
shp_sp["cd_setor"] = shp_sp["CD_SETOR"].astype(str).str.strip()
set_A = set(shp_sp.loc[shp_sp["cd_setor"].str.len() == 15, "cd_setor"])
print(f"|A| (setores no shapefile, {UF_SIGLA}): {len(set_A):,}")

Colunas do shapefile:
['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB']

Total de setores no shapefile (Brasil): 468,099
|A| (setores no shapefile, SP): 103,319


In [6]:
renda_raw = pd.read_csv(RENDA_CSV, sep=";", dtype=str, keep_default_na=False, na_filter=False)
print("Colunas do CSV de renda:")
print(list(renda_raw.columns)[:20], "...")
print(f"\nTotal de linhas (Brasil): {len(renda_raw):,}")
renda_raw["cd_setor"] = renda_raw["CD_SETOR"].astype(str).str.strip()
renda_sp = renda_raw.loc[
    (renda_raw["cd_setor"].str.len() == 15) & (renda_raw["cd_setor"].str.startswith(UF_COD))
].copy()
set_B = set(renda_sp["cd_setor"])

def parse_br(series):
    s = series.fillna("").astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

renda_sp["v06004_num"] = parse_br(renda_sp["V06004"])
set_B_star = set(renda_sp.loc[renda_sp["v06004_num"].notna(), "cd_setor"])
print(f"|B| (setores no CSV de renda, {UF_SIGLA}):              {len(set_B):,}")
print(f"|B*| (subset de B com V06004 numerico):        {len(set_B_star):,}")
print(f"|B − B*| (no CSV mas V06004 não numerico):     {len(set_B - set_B_star):,}")

Colunas do CSV de renda:
['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005'] ...

Total de linhas (Brasil): 458,772
|B| (setores no CSV de renda, SP):              100,928
|B*| (subset de B com V06004 numerico):        99,223
|B − B*| (no CSV mas V06004 não numerico):     1,705


In [7]:
with sqlite3.connect(WORK_SQLITE) as conn:
    rows = conn.execute("SELECT DISTINCT cd_setor FROM pares_logradouro").fetchall()
set_C = {r[0] for r in rows}
print(f"|C| (setores no CNEFE / pares_logradouro): {len(set_C):,}")

|C| (setores no CNEFE / pares_logradouro): 100,723


In [8]:
def summarize_set(name, s):
    return {"conjunto": name, "tamanho": len(s)}

venn = pd.DataFrame([
    summarize_set("A (shapefile)",            set_A),
    summarize_set("B (CSV renda)",            set_B),
    summarize_set("B* (B com V06004 num)",    set_B_star),
    summarize_set("C (CNEFE pares)",          set_C),
    summarize_set("A ∩ B",                    set_A & set_B),
    summarize_set("A ∩ C",                    set_A & set_C),
    summarize_set("B ∩ C",                    set_B & set_C),
    summarize_set("A ∩ B ∩ C",                set_A & set_B & set_C),
    summarize_set("C − B  (no CNEFE, fora do CSV renda)",   set_C - set_B),
    summarize_set("C − A  (no CNEFE, fora do shapefile)",   set_C - set_A),
    summarize_set("A − C  (no shapefile, fora do CNEFE)",   set_A - set_C),
    summarize_set("B − C  (no CSV renda, fora do CNEFE)",   set_B - set_C),
    summarize_set("C ∩ B − B*  (no CNEFE, B mas V06004 vazio)", (set_C & set_B) - set_B_star),
])
display(venn)

,conjunto,tamanho
0,A (shapefile),103319
1,B (CSV renda),100928
2,B* (B com V06004 num),99223
3,C (CNEFE pares),100723
4,A ∩ B,100928
5,A ∩ C,91944
6,B ∩ C,91097
7,A ∩ B ∩ C,91097
8,"C − B (no CNEFE, fora do CSV renda)",9626
9,"C − A (no CNEFE, fora do shapefile)",8779


## Etapa 3 — Caracterizar `C − B` (no CNEFE, fora do CSV de renda)

Esses são os setores que disparam `tem_renda=0` no output. Vamos olhar:

- município (top concentração)
- quantos endereços e CEPs eles representam
- se eles existem no shapefile (`C − B` ∩ A) ou não (`C − B` − A)

In [9]:
c_minus_b = set_C - set_B
print(f"|C − B|: {len(c_minus_b):,}")

with sqlite3.connect(WORK_SQLITE) as conn:
    df_cb = pd.read_sql_query(
        """
        SELECT cd_setor,
               MIN(cod_municipio) AS cod_municipio,
               COUNT(DISTINCT cep) AS qtd_ceps,
               COUNT(*) AS linhas_pares,
               SUM(qtd_enderecos) AS total_enderecos
        FROM pares_logradouro
        GROUP BY cd_setor
        """,
        conn,
    )
df_cb = df_cb.loc[df_cb["cd_setor"].isin(c_minus_b)].copy()
df_cb["esta_no_shapefile"] = df_cb["cd_setor"].isin(set_A)
print(f"Setores em C−B que estão no shapefile (A): {df_cb['esta_no_shapefile'].sum():,}")
print(f"Setores em C−B fora do shapefile:           {(~df_cb['esta_no_shapefile']).sum():,}")
print(f"Endereços impactados em C−B: {int(df_cb['total_enderecos'].sum()):,}")
display(df_cb.head(10))

|C − B|: 9,626
Setores em C−B que estão no shapefile (A): 847
Setores em C−B fora do shapefile:           8,779
Endereços impactados em C−B: 1,624,792


,cd_setor,cod_municipio,qtd_ceps,linhas_pares,total_enderecos,esta_no_shapefile
19,350010505000021,3500105,27,27,241,False
41,350010505000056,3500105,1,1,2,False
48,350010505000064,3500105,1,1,1,False
54,350010505000070,3500105,1,1,11,False
58,350010505000074,3500105,2,2,12,False
60,350010505000076,3500105,1,2,8,False
63,350010505000079,3500105,4,4,27,False
70,350010505000091,3500105,2,4,8,False
71,350010505000092,3500105,1,1,1,True
75,350010505000097,3500105,6,8,127,False


In [10]:
top_mun = (
    df_cb.groupby("cod_municipio")
    .agg(setores=("cd_setor", "nunique"),
         total_enderecos=("total_enderecos", "sum"))
    .sort_values("setores", ascending=False)
    .head(20)
)
display(top_mun)

,setores,total_enderecos
cod_municipio,,
3550308,1464,172714
3509502,630,131146
3522208,194,32874
3525904,181,24971
3547809,155,27296
3549904,143,18116
3548708,139,18272
3515004,120,14647
3518800,114,21101


## Etapa 4 — Carregar `CD_TIPO` do shapefile e cruzar

A pipeline atual não carrega o tipo de setor. Vamos ver:

- quais colunas de tipo existem no shapefile (`CD_TIPO`, `NM_TIPO`, `TIPO`, ...)
- distribuição de tipo dentro de `C − B`
- distribuição de tipo dentro de `C ∩ B − B*` (sigilo disfarçado)
- distribuição de tipo dentro de `A − C` (perda silenciosa: setor no shapefile que não aparece no output)

In [11]:
tipo_cols = [c for c in shp_sp.columns if "TIPO" in c.upper() or "SITUACAO" in c.upper() or "SITUAÇÃO" in c.upper()]
print("Possíveis colunas de tipo/situação no shapefile:")
print(tipo_cols)
if tipo_cols:
    for col in tipo_cols:
        print(f"\nDistribuição global em {UF_SIGLA} de {col}:")
        display(shp_sp[col].astype(str).value_counts(dropna=False).head(20))

Possíveis colunas de tipo/situação no shapefile:
['SITUACAO', 'CD_TIPO']

Distribuição global em SP de SITUACAO:


SITUACAO
Urbana    94024
Rural      9047
NaN         248
Name: count, dtype: int64


Distribuição global em SP de CD_TIPO:


CD_TIPO
0    91996
1     7995
4     2715
7      217
6      178
5       77
9       64
3       27
8       26
2       24
Name: count, dtype: int64

In [12]:
if not tipo_cols:
    print("Sem colunas de tipo no shapefile — pular cruzamento.")
else:
    tipo_col = tipo_cols[0]  # primeira encontrada (provavelmente CD_TIPO)
    tipo_lookup = shp_sp.set_index("cd_setor")[tipo_col].astype(str)
    grupos = {
        "C − B (sem renda no output)":          set_C - set_B,
        "C ∩ B − B* (sigilo disfarçado)":       (set_C & set_B) - set_B_star,
        "A − C (perda silenciosa, fora output)": set_A - set_C,
        "A ∩ C ∩ B*  (caso saudável)":           set_A & set_C & set_B_star,
    }
    for nome, grupo in grupos.items():
        sub = tipo_lookup.loc[tipo_lookup.index.isin(grupo)]
        total = len(grupo)
        match = len(sub)
        print(f"\n=== {nome}  |grupo|={total:,}  com_tipo_no_shapefile={match:,} ===")
        if match:
            display(sub.value_counts(dropna=False).head(15))


=== C − B (sem renda no output)  |grupo|=9,626  com_tipo_no_shapefile=847 ===


SITUACAO
Urbana    633
Rural     210
NaN         4
Name: count, dtype: int64


=== C ∩ B − B* (sigilo disfarçado)  |grupo|=1,126  com_tipo_no_shapefile=1,126 ===


SITUACAO
Urbana    677
Rural     449
Name: count, dtype: int64


=== A − C (perda silenciosa, fora output)  |grupo|=11,375  com_tipo_no_shapefile=11,375 ===


SITUACAO
Urbana    7523
Rural     3608
NaN        244
Name: count, dtype: int64


=== A ∩ C ∩ B*  (caso saudável)  |grupo|=89,971  com_tipo_no_shapefile=89,971 ===


SITUACAO
Urbana    85191
Rural      4780
Name: count, dtype: int64

## Etapa 5 — Inspecionar o CSV de renda cru

Em `parse_br_number_series` da pipeline, qualquer string não numérica vira `NaN`. Isso significa que **sigilo** e **célula vazia** colapsam num mesmo NULL no SQLite. Aqui olhamos o CSV cru para ver os marcadores reais.

In [13]:
v06004_raw = renda_sp["V06004"].astype(str).str.strip()
v06004_num = renda_sp["v06004_num"]
nao_numericos = v06004_raw.loc[v06004_num.isna()]
print(f"Linhas com V06004 não numerico em {UF_SIGLA}: {len(nao_numericos):,}")
print("\nMarcadores únicos encontrados (top 20):")
display(nao_numericos.value_counts(dropna=False).head(20))

Linhas com V06004 não numerico em SP: 1,705

Marcadores únicos encontrados (top 20):


V06004
X    1697
,       8
Name: count, dtype: int64

In [14]:
# Para os setores em B com V06004 ausente, verificar se as outras variáveis (V06001..V06005) também estão vazias.
vars_renda = ["V06001", "V06002", "V06003", "V06004", "V06005"]
for v in vars_renda:
    renda_sp[f"{v}_num"] = parse_br(renda_sp[v])

sigilo = renda_sp.loc[renda_sp["v06004_num"].isna()].copy()
print(f"Setores em B−B* (V06004 vazio): {len(sigilo):,}")
ausencia_total = sigilo[[f'{v}_num' for v in vars_renda]].isna().all(axis=1)
print(f"  com TODAS V06001..V06005 vazias: {int(ausencia_total.sum()):,}")
print(f"  com pelo menos uma outra V06xxx preenchida: {int((~ausencia_total).sum()):,}")
display(sigilo[["cd_setor"] + vars_renda].head(10))

Setores em B−B* (V06004 vazio): 1,705
  com TODAS V06001..V06005 vazias: 1,697
  com pelo menos uma outra V06xxx preenchida: 8


,cd_setor,V06001,V06002,V06003,V06004,V06005
259289,350010505000032,X,X,X,X,X
259302,350010505000051,X,X,X,X,X
259315,350010505000071,X,X,X,X,X
259318,350010505000075,X,X,X,X,X
259325,350010505000089,X,X,X,X,X
259351,350010505000131,X,X,X,X,X
259352,350010505000134,X,X,X,X,X
259510,350055005000018,X,X,X,X,X
259565,350070905000039,X,X,X,X,X
259566,350070905000040,X,X,X,X,X


## Etapa 6 — Síntese

Consolidação das causas com contagens de setores. Esta tabela é o produto final do diagnóstico — base para decidir como evoluir o output da pipeline.

In [15]:
categorias = [
    ("1. Saudável (no shapefile, no CNEFE, com renda numerica)", set_A & set_C & set_B_star),
    ("2. CNEFE + CSV mas V06004 vazio (sigilo / sem domic.)",    (set_C & set_B) - set_B_star),
    ("3. CNEFE mas ausente do CSV de renda (C − B)",              set_C - set_B),
    ("   3a.  e existe no shapefile (C − B) ∩ A",                (set_C - set_B) & set_A),
    ("   3b.  e NÃO existe no shapefile (C − B) − A",            (set_C - set_B) - set_A),
    ("4. No shapefile mas fora do CNEFE — perda silenciosa",     set_A - set_C),
    ("5. No CSV renda mas fora do CNEFE",                        set_B - set_C),
]
sintese = pd.DataFrame({
    "categoria": [nome for nome, _ in categorias],
    "setores":   [len(s) for _, s in categorias],
})
display(sintese)

print("\nObservações esperadas:")
print("- Categoria 1 deve ser a maioria.")
print("- Categorias 2 e 3 explicam o 'sem renda' visível no output (tem_renda=0 e V06004 NULL).")
print("- Categoria 4 é INVISÍVEL no output atual (setor do shapefile sem endereço CNEFE).")
print("- Categoria 3b, se >0, indica códigos de setor estranhos no CNEFE.")

,categoria,setores
0,"1. Saudável (no shapefile, no CNEFE, com renda...",89971
1,2. CNEFE + CSV mas V06004 vazio (sigilo / sem ...,1126
2,3. CNEFE mas ausente do CSV de renda (C − B),9626
3,3a. e existe no shapefile (C − B) ∩ A,847
4,3b. e NÃO existe no shapefile (C − B) − A,8779
5,4. No shapefile mas fora do CNEFE — perda sile...,11375
6,5. No CSV renda mas fora do CNEFE,9831



Observações esperadas:
- Categoria 1 deve ser a maioria.
- Categorias 2 e 3 explicam o 'sem renda' visível no output (tem_renda=0 e V06004 NULL).
- Categoria 4 é INVISÍVEL no output atual (setor do shapefile sem endereço CNEFE).
- Categoria 3b, se >0, indica códigos de setor estranhos no CNEFE.


## Etapa 7 — Por que 8.779 setores do CNEFE não estão na CD2022?

A categoria **3b** da síntese (`C − A`) tem 8.779 setores: o CNEFE traz o código e o shapefile CD2022 não. Aqui investigamos **onde** mora a divergência.

O `cd_setor` segue o padrão IBGE de 15 dígitos:

```
UF (2) + Município (5) + Distrito (2) + Subdistrito (2) + Setor (4)
```

Decompondo, podemos verificar se a divergência aparece já no **município**, no **distrito**, no **subdistrito**, ou somente no **número do setor** (último bloco). Cada caso aponta para uma causa diferente:

- **Município ausente** → código inválido / sujeira.
- **Distrito ou subdistrito ausente** → diferença de codificação administrativa entre o CNEFE e a malha CD2022.
- **Só o número do setor difere** → versão diferente da malha (CD2010 vs CD2022, por ex.) ou setor desmembrado/incorporado depois.

In [16]:
# Decompor cd_setor (15 dígitos) em UF + Mun(5) + Dist(2) + Subdist(2) + Setor(4).
def parts(series):
    return pd.DataFrame({
        "uf":      series.str[:2],
        "mun":     series.str[2:7],
        "dist":    series.str[7:9],
        "subdist": series.str[9:11],
        "setor":   series.str[11:15],
    }, index=series.index)


c_minus_a = set_C - set_A
print(f"|C − A| (setores no CNEFE fora do shapefile): {len(c_minus_a):,}")

cma_df = pd.DataFrame({"cd_setor": sorted(c_minus_a)})
cma_df = pd.concat([cma_df, parts(cma_df["cd_setor"])], axis=1)
cma_df["cod_municipio"]    = cma_df["uf"] + cma_df["mun"]
cma_df["mun_dist"]         = cma_df["cod_municipio"] + cma_df["dist"]
cma_df["mun_dist_subdist"] = cma_df["mun_dist"] + cma_df["subdist"]

# Construir a referência a partir do shapefile (SP, set A).
shp_ref = shp_sp[["cd_setor"]].copy().reset_index(drop=True)
shp_ref = pd.concat([shp_ref, parts(shp_ref["cd_setor"])], axis=1)
shp_ref["cod_municipio"]    = shp_ref["uf"] + shp_ref["mun"]
shp_ref["mun_dist"]         = shp_ref["cod_municipio"] + shp_ref["dist"]
shp_ref["mun_dist_subdist"] = shp_ref["mun_dist"] + shp_ref["subdist"]

mun_in_shp           = set(shp_ref["cod_municipio"])
mun_dist_in_shp      = set(shp_ref["mun_dist"])
mun_dist_subdist_shp = set(shp_ref["mun_dist_subdist"])

cma_df["mun_ok"]      = cma_df["cod_municipio"].isin(mun_in_shp)
cma_df["mun_dist_ok"] = cma_df["mun_dist"].isin(mun_dist_in_shp)
cma_df["mds_ok"]      = cma_df["mun_dist_subdist"].isin(mun_dist_subdist_shp)

print("\nPreview dos códigos órfãos decompostos:")
display(cma_df.head(10))

|C − A| (setores no CNEFE fora do shapefile): 8,779

Preview dos códigos órfãos decompostos:


,cd_setor,uf,mun,dist,subdist,setor,cod_municipio,mun_dist,mun_dist_subdist,mun_ok,mun_dist_ok,mds_ok
0,350010505000021,35,00105,05,00,0021,3500105,350010505,35001050500,True,True,True
1,350010505000056,35,00105,05,00,0056,3500105,350010505,35001050500,True,True,True
2,350010505000064,35,00105,05,00,0064,3500105,350010505,35001050500,True,True,True
3,350010505000070,35,00105,05,00,0070,3500105,350010505,35001050500,True,True,True
4,350010505000074,35,00105,05,00,0074,3500105,350010505,35001050500,True,True,True
5,350010505000076,35,00105,05,00,0076,3500105,350010505,35001050500,True,True,True
6,350010505000079,35,00105,05,00,0079,3500105,350010505,35001050500,True,True,True
7,350010505000091,35,00105,05,00,0091,3500105,350010505,35001050500,True,True,True
8,350010505000097,35,00105,05,00,0097,3500105,350010505,35001050500,True,True,True
9,350010505000098,35,00105,05,00,0098,3500105,350010505,35001050500,True,True,True


In [17]:
# Classificar onde mora a divergência.
def classify(row):
    if not row["mun_ok"]:
        return "1. municipio nao existe no shapefile"
    if not row["mun_dist_ok"]:
        return "2. municipio OK, distrito nao existe"
    if not row["mds_ok"]:
        return "3. municipio+distrito OK, subdistrito nao existe"
    return "4. mun+dist+subdist OK — somente o numero do setor difere"

cma_df["divergencia"] = cma_df.apply(classify, axis=1)
div_counts = cma_df["divergencia"].value_counts().sort_index()
print("Onde mora a divergencia entre codigo CNEFE e shapefile CD2022:")
display(div_counts.to_frame(name="setores"))

print("\nCheck — UFs presentes em C − A (esperado: somente '35'):")
display(cma_df["uf"].value_counts())

Onde mora a divergencia entre codigo CNEFE e shapefile CD2022:


,setores
divergencia,
4. mun+dist+subdist OK — somente o numero do setor difere,8779



Check — UFs presentes em C − A (esperado: somente '35'):


uf
35    8779
Name: count, dtype: int64

In [18]:
# Top municípios afetados pela categoria 3b.
top_mun_cma = (
    cma_df.groupby("cod_municipio")
    .agg(setores_orfaos=("cd_setor", "count"),
         mun_existe_no_shp=("mun_ok", "first"))
    .sort_values("setores_orfaos", ascending=False)
    .head(20)
)
print("Top 20 municipios com mais setores orfaos (C − A):")
display(top_mun_cma)

# Distribuição: quantos municípios distintos a categoria 3b cobre, e quantos
# desses municípios existem no shapefile.
mun_unicos = cma_df["cod_municipio"].nunique()
mun_existentes = cma_df.loc[cma_df["mun_ok"], "cod_municipio"].nunique()
print(f"\nMunicipios distintos em C − A: {mun_unicos}")
print(f"  desses, que existem no shapefile: {mun_existentes}")
print(f"  desses, ausentes do shapefile:    {mun_unicos - mun_existentes}")

Top 20 municipios com mais setores orfaos (C − A):


,setores_orfaos,mun_existe_no_shp
cod_municipio,,
3550308,1293,True
3509502,613,True
3522208,191,True
3525904,179,True
3547809,129,True
3549904,128,True
3548708,127,True
3515004,103,True
3518800,92,True



Municipios distintos em C − A: 564
  desses, que existem no shapefile: 564
  desses, ausentes do shapefile:    0


In [19]:
# Exemplo concreto: pegar o município em que o CNEFE traz mais setores órfãos
# (entre os municípios que existem no shapefile) e comparar dist+subdist
# CNEFE (apenas dos códigos órfãos) vs shapefile (todos os do município).

cand = (
    cma_df.loc[cma_df["mun_ok"]]
    .groupby("cod_municipio")
    .size()
    .sort_values(ascending=False)
)
if len(cand) == 0:
    print("Nenhum municipio com codigos orfaos onde o municipio exista no shapefile.")
else:
    exemplo_mun = cand.index[0]
    print(f"Exemplo: municipio {exemplo_mun} — {cand.iloc[0]:,} setores orfaos no CNEFE")

    cnefe_ds = (
        cma_df.loc[cma_df["cod_municipio"] == exemplo_mun, ["dist", "subdist"]]
        .value_counts()
        .reset_index(name="setores_orfaos_cnefe")
    )
    shp_ds = (
        shp_ref.loc[shp_ref["cod_municipio"] == exemplo_mun, ["dist", "subdist"]]
        .value_counts()
        .reset_index(name="setores_no_shapefile")
    )
    comparacao = (
        pd.merge(cnefe_ds, shp_ds, on=["dist", "subdist"], how="outer")
        .fillna(0)
    )
    for col in ["setores_orfaos_cnefe", "setores_no_shapefile"]:
        comparacao[col] = comparacao[col].astype(int)
    comparacao = comparacao.sort_values(["dist", "subdist"]).reset_index(drop=True)
    print(f"\nDistritos/subdistritos em {exemplo_mun}:")
    print("  setores_orfaos_cnefe  = quantos códigos C−A o CNEFE tem nessa combinação dist+subdist")
    print("  setores_no_shapefile  = quantos setores o shapefile tem na mesma combinação\n")
    display(comparacao)

Exemplo: municipio 3550308 — 1,293 setores orfaos no CNEFE

Distritos/subdistritos em 3550308:
  setores_orfaos_cnefe  = quantos códigos C−A o CNEFE tem nessa combinação dist+subdist
  setores_no_shapefile  = quantos setores o shapefile tem na mesma combinação



,dist,subdist,setores_orfaos_cnefe,setores_no_shapefile
0,01,00,0,174
1,02,00,0,134
2,03,00,26,170
3,04,00,1,163
4,05,00,2,205
5,06,00,4,116
6,07,00,16,256
7,08,00,0,107
8,09,00,2,92
9,10,00,7,133


## Conclusão da Etapa 7 — resultado da execução

**Hipótese 2 (versão diferente da malha) confirmada inequivocamente.**

Os 8.779 setores órfãos (`C − A`) **todos** caíram na categoria 4: `mun + dist + subdist` batem com o shapefile CD2022; apenas o **último bloco de 4 dígitos (número do setor)** é diferente.

Resumo dos números observados:

- UF: 100% `35` (sanity OK).
- Municípios: 564 distintos, **todos existem** no shapefile (zero municípios inválidos).
- Distrito + subdistrito: todas as combinações observadas em `C − A` existem no shapefile.
- Última parte do código (`setor`, 4 dígitos): é o único bloco que difere.

Concentração: capital de São Paulo (`3550308`) tem 1.293 órfãos (~15% do total), Campinas (`3509502`) 613. Em SP-capital, **quase todo distrito** tem alguns órfãos, com variação 0–151. Caso extremo: distrito 52 da capital tem 48 órfãos vs apenas 76 setores no shapefile (~63% de divergência), provável reorganização pesada entre versões.

**Causa**: o CNEFE 2022 carrega endereços cujos `cd_setor` correspondem a setores de uma **versão anterior** da malha (provavelmente CD2010, ou de um lote intermediário antes da consolidação CD2022). O IBGE renumera, desmembra e funde setores entre censos — os setores antigos não foram migrados para a numeração CD2022 nesses ~8.779 registros.

## Próximos passos

Resumo do que o diagnóstico (Etapas 1–7) entregou para SP:

| Categoria | Setores | Endereços | % Endereços |
|---|---:|---:|---:|
| Saudável (renda numérica) | 89.971 | 21.307.605 | 92,83% |
| Sigilo IBGE (marcador `"X"`) | 1.126 | 21.328 | 0,09% |
| Setor órfão CNEFE — versão antiga (cat. 3b) | 8.779 | ~1.6M | ~7% |
| C∩A mas fora do CSV de renda (cat. 3a) | 847 | (parte do 7%) | |
| Perda silenciosa (A − C, não aparece no output) | 11.375 | — | — |

### Três caminhos para evoluir a pipeline

| Opção | Esforço | O que entrega |
|---|---|---|
| **A. Marcar motivo no output** | Baixo | Adicionar coluna `motivo_sem_renda` ∈ `{tem_renda, sigilo_ibge, setor_orfao_cnefe, setor_ausente_do_csv}`. Carregar `SITUACAO` e `CD_TIPO` do shapefile e expor `tipo_setor` no output. Distinguir o marcador `"X"` em `parse_br_number_series` para não colapsar sigilo com NaN. |
| **B. Reconciliar via tabela IBGE** | Médio | Baixar a tabela de compatibilização CD2010 ↔ CD2022 do IBGE e fazer mapping `setor_antigo → setor_novo` (1:1, 1:N, N:1). Recupera renda na maioria dos órfãos com link direto. |
| **C. Reconciliação espacial** | Alto | Calcular geometria do setor antigo via CEP/logradouro, projetar no shapefile CD2022 para inferir o setor novo. Funciona sem a tabela do IBGE mas é caro e tem ruído. |

### Recomendação para o próximo ciclo

1. Implementar **A** primeiro — é a mudança de menor risco e maior valor informacional. A pipeline continua reprodutível; o consumidor passa a saber exatamente *por que* um setor ficou sem renda.
2. Em paralelo, pesquisar disponibilidade da **tabela de compatibilização IBGE 2010↔2022** para viabilizar **B**.
3. Repetir o diagnóstico em outra UF (ex.: RJ) para validar se o padrão de SP se mantém.
4. Decisão de design pendente: queremos expor a **categoria 4** (`A − C`, perda silenciosa, 11.375 setores) no output? Isso exigiria mudar o `LEFT JOIN` da `final_query` para `FULL OUTER` (ou union com setores do shapefile sem CNEFE).